# Markov State Model to describe a methyle dynamics

The objectives of this tutorial is to :

1. Discover some basics of `pyEmma`, the package that we will use to make MSM for MD simulation:
    * to load a MD trajectory,
    * to measure a dihedral angle as a function of time. 
    *

2. Generate your first MSM : 
    * define the states,
    * define the timelaps,
    * measure the transition matrix,
    *

3. Test if the protein dynamics is well reproduced by your Markov State Model. 



<font color='orange'>

**ATTENTION**  
To run this notebook, you have to 
1. trust the notebook, 
2. choose the jupyter kernel named 'msm' in which the packages are installed. 
To do so, in the jupyter notebook menu, look for `Kernel>Change Kernel>msm`
3. read the notebook and make changes where you see the keyword `TOCOMPLETE`
</font>



In [ ]:
# first import the packages
import pyemma
import mdshare
from pyemma.util.contexts import settings
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pexpect
import os
import random


## Import the MD trajectories from a databank, using mdshare

We are going to download the trajectories of a pentapeptide.
These trajectories are available at the url
https://markovmodel.github.io/mdshare/pentapeptide/

There are :
- 1 pdb file that contains the types of the atoms, and one conformations.
- 25 trajectories available...you can choose the one you want (in the example, we choose randomly)

In the following cell, you use the command `mdshare.fetch` 
to download them in a directory  called `data`.
Then, you open vmd to visualize the pdb file.

Here, we define also a directory to gather the pictures.


In [ ]:
# define a directory for outputs
outdir = "Figures/"
if not os.path.exists(outdir):
    os.makedirs(outdir)

In [ ]:
# import a  MD simulation from a databank

# chose a random trajectory among the one available, numbered from 0 to 24
chosen = str(random.randint(0, 24)).zfill(2)


#mdshare.catalogue()
pdb = mdshare.fetch('pentapeptide-impl-solv.pdb', working_directory='data',show_progress=False)
files = mdshare.fetch(f'pentapeptide-{chosen}-500ns-impl-solv.xtc', working_directory='data',show_progress=False)
print(pdb)
print(files)


# visualise the peptide using vmd
Vizualize = False
if Vizualize : 
    vmd = pexpect.spawn("vmd") # Ou "/chemin/vers/le/programme"
    vmd.expect("vmd > ") # Le paramètre est le prompt du programme
    vmd.sendline(f"mol load pdb {pdb} xtc {files}")
    vmd.expect("vmd > ")
    vmd.sendline("display projection orthographic")
    vmd.expect("vmd > ")
    vmd.sendline("color Display Background black")
    vmd.expect("vmd > ")



<font color='red'>

**QUESTIONS**  
</font>

**We shall study the kintics of the rotation around the $C_A - C_B$** bond.

1. After you have run the previous cell, verify that you have downloaded the files.
2. Visualize the pentapeptide in the first, extended conformation.
3. Localise the $NH_2$-terminus, the $CO_2H$ terminus, the 4 peptide bonds. Are the peptide bond planar ?
4. Localise the Alanine in the center, residue number 3, with its methyle ($CH_3$). Note the $C_A$ and the $C_B$ carbon atom. The  $C_A$ belongs to the main chain. The $C_B$ is the carbon of the methyle group.
5. Look in the `data/pentapeptide-impl-solv.pdb` file.
What are the indices of the atoms HA, CA, CB, HB1 ?
In the next cell, you'll have to define the list of these indices.



## Now measure some dihedral angles.
We shall use the `feature` notion of `pyemma`. It is a description of the molecules that one can define. 

The possible descriptors are explained  in this website
http://www.emma-project.org/latest/api/generated/pyemma.coordinates.featurizer.html


Here, we shall define and measure the time evolution of :
1. a dihedral describing the rotation of the methyle of the middle Alanine. 
We use the index list `PDB_indices_list` to define the atoms. 
2. all the phi and psi backbone dihedrals

The `description` of the features provide a list  of the name of the features.

In [ ]:
# here the list of indices for the atoms that define our dihedrals
# REMOVE THIS FOR STUDENTS
#ATOM     47  CA  ALA X   3      19.920  18.890  19.780  0.00  0.00            
#ATOM     48  HA  ALA X   3      18.930  18.980  20.200  0.00  0.00            
#ATOM     49  CB  ALA X   3      19.850  18.320  18.340  0.00  0.00            
#ATOM     50  HB1 ALA X   3      19.350  17.330  18.330  0.00  0.00            

# PDB indices
# be careful about the ORDER to define the rotation around the CA-CB bond.
# 
#PDB_indices_list = TOCOMPLETE
PDB_indices_list = [48,47,49,50]

# in Pyemma, the indices begin at 0. 
# So we have to shift the PDB_indices by (-1) for the following.
PYEMMA_indices_list = [i-1 for i in PDB_indices_list]


# Use the topology of the PDB file
torsions_feat = pyemma.coordinates.featurizer(pdb)

# define as observable several dihedrals
# 1. methyle angles of the backbone)
torsions_feat.add_dihedrals([PYEMMA_indices_list],cossin=False, )
#2. the phi and psi angle of the backbone
torsions_feat.add_backbone_torsions(cossin=False, periodic=True)


# load the trajectory files into pyemma and measure the dihedrals
torsions_data = pyemma.coordinates.load(files, features=torsions_feat)
labels = ['backbone\ntorsions']


# describe the format the data
print(torsions_feat.describe())
number_features = len(torsions_feat.describe())
number_timesteps = np.shape(torsions_data)[0]

print(f' shape of the data = {np.shape(torsions_data)}')
print(f' number of features  = {np.shape(torsions_data)[1]}')
print(f' number of timesteps in the trajectory  = {np.shape(torsions_data)[0]}')


# if several trajectories were loaded 
#torsions_data_concatenated = np.concatenate(torsions_data)
#print(np.shape(torsions_data_concatenated))



Plot the time evolution and the probability distribution of the data

In [ ]:

# Plot the time evolution of the feature 

# we know that the data are stored every 0.1 ns
timestep = 0.1

fig, axes = plt.subplots(number_features, 1, figsize=(12, number_features), sharex=True)
for i in np.arange(number_features):
    tor = torsions_data[:,i]
    x = timestep*np.arange(len(tor))
    ax = axes.flat[i]
    ax.plot(x, tor)
    ax.set_ylabel(torsions_feat.describe()[i])
axes[-1].set_xlabel('time / ns')
fig.tight_layout()
fig.savefig(f'{outdir}/time_series_all_features.png')


In [ ]:

# Plot  the histogramm
fig, ax = pyemma.plots.plot_feature_histograms(torsions_data,feature_labels=torsions_feat.describe())
fig.savefig(f'{outdir}/histogram_all_features.png')



<font color='red'>

**QUESTIONS**  
</font>

1. Look at the time evolution and the distribution of the dihedrals. 
2. Which one seems the easiest to define a few separated states for a Markov State Model ? 
3. How many states do you recognize ? *Hint : remember that a dihedral is periodic with periodicity of $2\pi$ radians.* 


In [ ]:
# after this cell, we focus one dihedral defined by its label, called "chosen_label"

#chosen_label = 'PSI 0 LEU 2'
chosen_label = 'DIH: ALA 3 HA 47 - ALA 3 CA 46 - ALA 3 CB 48 - ALA 3 HB1 49 '
#chosen_label = TOCOMPLETE

chosen_index =  torsions_feat.describe().index(chosen_label)
chosen_label = chosen_label.replace(' ','').replace(':','.')

# selection the torsion data for the chosen label
tor = torsions_data[:,chosen_index]

# plot the chosen data again
fig, axes = plt.subplots(1, 1, figsize=(20, number_features), sharex=True)
x = timestep*np.arange(len(tor))
axes.plot(x, tor)
axes.set_ylabel('dihedral / radian')
axes.set_xlabel('time / ns')
axes.set_title(f'{chosen_label} as a function of time')
fig.savefig(f"""{outdir}/{chosen_label}_timeserie.png""")


It is easier to analyse the data if a single state corresponds to a single peak in the probability distribution.
To have this property, we shall wrap the data in an interval different from $[-\pi,\pi]$. 
Here, an interval $[-2,-2+\pi]$ seems nice.

<font color='red'>

**QUESTIONS**  
</font>
1. Define a wrapping function `wrap_angle`.
2. Apply it to the `tor` data.
3. Plot the new distribution and time evolution. 
4. Do you have now all the states within one peak of the distribution ? 
5. Define the number of states that you want to use for your Markov State Model.


In [ ]:

def wrap_angle(x,min,period):
    """define a function to wrap a periodic observable 'x'
    of periodicity 'period'
    the final angle between  is [min and min+period]
    the algorithm is only valid if 'min' is within the initial range!
    """
    if x < min :
        x = x+period
    if x > min+period :
        x = x-period
    return x


In [ ]:
# apply the wrap function to the torsion data.
wrap_tor = [ wrap_angle(a,-2,2*np.pi) for a in tor] 

# plot the time evolution 
fig, axes = plt.subplots(1, 1, figsize=(12, 4), sharex=True)
x = timestep*np.arange(len(tor))
axes.plot(x, wrap_tor)
axes.set_ylabel('dihedral angle / radian')
axes.set_xlabel('time / ns')
axes.set_title(f'{chosen_label} as a function of time')
fig.savefig(f"""{outdir}/{chosen_label}_timeserie_wraped.png""")
plt.show()
plt.close()

# plot the distribution
nbins = 100
step = (np.max(wrap_tor)-np.min(wrap_tor))/nbins
bins = np.arange(np.min(wrap_tor),np.max(wrap_tor),step)

fig, axes = plt.subplots(1, 1, figsize=(6, 4), sharex=True)
plt.hist(wrap_tor,bins=bins, density=True)
axes.set_xlabel(f'{chosen_label} values')
axes.set_title(f'Probability distribution')
fig.savefig(f"""{outdir}/{chosen_label}_histogram_wraped.png""")
plt.show()
plt.close()


In [ ]:
# number of states
ncluster = 3
#ncluster = TOCOMPLETE

### Separate the data into states

Here, you must separate the data into `ncluster` states. You have to :
*  find the centers of clusters
*  attribute each data to a given cluster (the "closest one").

You can do it per hand, or use a clustering algorithm. In this notebook, two functions have been writen :
1. `attribute_3cluster(data,limit1,limit2)` : to do it per hand, you can look at the distribution and define 2 limits in the distribution.
2. `clusterize_kmedoids(data,ncluster)` , using the algorithm of the same name.


For the first try, we try using the `attribute_3cluster` function. 

In [ ]:
def attribute_3cluster(data,limit1,limit2):
    cluster = []
    for d in data:
        if d < limit1 :
            cluster.append(0)
        elif d < limit2 :
            cluster.append(1)
        else :
            cluster.append(2)
            # cluster.append(TOCOMPLETE)

    return cluster

First try it by hand. Define the limits `mylimit1` and `mylimit2` between the clusters and see the repartition into clusters. The output of the function if a serie of states.

In [ ]:
mylimit1 = 0.0
mylimit2 = 2.0

# define the limits
#mylimit1 = TOCOMPLETE
#mylimit2 = TOCOMPLETE

# attributes the clusters
mytraj3=  attribute_3cluster(wrap_tor,mylimit1,mylimit2)

# plot the cluster as a funtion of time
fig, axes = plt.subplots(2, 1, figsize=(12,10), sharex=True)
axes[0].plot(wrap_tor)
axes[0].set_ylabel("angle (radian)")
axes[0].set_title(f'{chosen_label} as a function of time')
axes[1].plot(mytraj3,'k.')
axes[1].set_ylabel("State number")
axes[1].set_xlabel('steps')
axes[1].set_title(f'State as a function of time (3 states)')
fig.savefig(f'{outdir}/{chosen_label}_attribute_clusters.png')




In [ ]:


fig, axes = plt.subplots(1, 1, figsize=(6,4), sharex=True)
for nc in range(ncluster):
    selected = [t for t,c in zip(wrap_tor,mytraj3) if c == nc]
    if len(selected) > 0 :
        plt.hist(selected,bins=50, density=True,label=f'cluster {nc}')
        #plt.hist(selected,bins=50, density=True,label=TOCOMPLETE)
       
axes.set_ylabel(torsions_feat.describe()[chosen_index])
axes.set_xlabel(f'{chosen_label} values')
axes.set_title(f'Probability distribution')
fig.legend()
fig.savefig(f"""{outdir}/{chosen_label}_histogram_attributed_ncluster{ncluster}.png""")

plt.show()
plt.close()

### Define the transition matrix


The transition matrix $T_{ij}$  is the probability to jump from $i$ to $j$, 
given that on is in the state $i$.

First, we count the  number of transitions from state $i$ to $j$ noted  $C_{ij}$ .
Then, we renormalize it by the number of time when $i$ is observed $N_i = Sum_j C_{ij}$.

So the Transition probability of the Markoc Model is defined as:
$T_{ij} = C_{ij}/N_i$.

<font color='red'>

**QUESTIONS**  

</font>

1. In the following cell, correct the  lines with `TOCOMPLETE` hints.


In [ ]:
def count_transition_matrix(trajectory,lag) :
    """ 
    function that counts the  number of transitions
    from state i to j. C_{ij}

    then, it renormalize it by the number
    of time i i is observed 
    N_i = Sum_j C_{ij}

    T_{ij} = C_{ij}/N_i

    * trajectory is a list of clusternumbers (evolution over time).    
    * lag is a list of lagtimes.
    """
    #determine the number of states in the trajectory
    nstates = int(np.max(trajectory)-np.min(trajectory)+1)
    # determine the number of timestep in the trajectory
    nsteps = len(trajectory)

    # initialise transition matrix, and count transitions between states
    transitions = np.zeros(dtype=float,shape=(nstates,nstates))
    for ti in np.arange(nsteps-lag):
    #for ti in np.arange(TOCOMPLETE):
        transitions[trajectory[ti]][trajectory[ti+lag]] += 1

    # renormalize according to initial state
    transitions_rescaled = np.zeros(dtype=float,shape=(nstates,nstates))
    for i in np.arange(nstates):
        total_count = np.sum(transitions[i][:])
        if total_count > 0 :
            for  j in np.arange(nstates) :
                transitions_rescaled[i][j]  = transitions[i][j] /total_count 
                #transitions_rescaled[i][j]  = TOCOMPLETE

    return(nstates,transitions,transitions_rescaled)


<font color='red'>

**QUESTIONS**  

</font>

1. Apply  the function `count_transition_matrix` to the `mytraj3` data for a time-lag of 1 timesteps.
2. During a time 0.1ns (=1timestep), is it more probable to jump to a other cluster, or to stay in the same cluster ?
3. Are the three clusters equivalent ? Why ?
4. for a large timelag, do the probability to jump in another cluster depend on the initial cluster ?

In [ ]:
trajectory = mytraj3
#lag = TOCOMPLETE
lag = 1
print(f'\n for lag = {lag}, transition matrix is ')
print(count_transition_matrix(trajectory,lag)[2])

#lag = TOCOMPLETE
lag = 30
print(f'\n for lag = {lag}, transition matrix is ')
print(count_transition_matrix(trajectory,lag)[2])




## Test Markovianity

To test the Markovianity of the data at the lag time $\tau$, 
we shall compare the predictions for the jumping probability
 at time $k\times\tau$ , ie $T_{ij}(k\tau)$
1. the Markov State Model obtained with a lag time $\tau$ : $T_{ij}(k\tau)$
2. the observation from the MD data.

This test is called the Chapman-Kolmogorov (CK) test.

**QUESTIONS**  

</font>

1. If we have a MSM  obtained at lag time $\tau$, using $T_{ij}(\tau)$. 

How can we deduce the jumping probability at time $2\tau$, ie $T_{ij}(2\tau)$ ?

How can we deduce the jumping probability at time $k\tau$, ie $T_{ij}(k\tau)$ ?

2. Complete the following cell on the lines with `TOCOMPLETE`  

In [ ]:

def CKtest(trajectory,laglist=[1],outdir="./"):
    """

    Make figures that contain the CK test ie.
    the comparison of $T_{ij}(k\tau)$  with $T_{ij}(\tau)^k$
    for each {ij} couples 
    and for various k

    input : 
        trajectory : list of cluster number as a function of time
        laglist : list of timelags $\tau$ to be tested
        outdir : directory in which the pictures are plotted
    """
    for lag in laglist:
        #lag = int(3)

        # measure the transition matrix
        n,Tc,T = count_transition_matrix(trajectory= trajectory,lag=lag)


        # create a plot with {sstates x nstates} subfigure for each of the {ij} couples
        fig, axes = plt.subplots(n,n,figsize=(10,10), sharex=True, sharey=True)

        # define a reasonable value for the maximum k
        kmax =  min(int(len(trajectory)/lag),50)

        for k in np.arange(1,kmax):
            newlag=int(k*lag)


            # calculate the two matrices $T_{ij}(k\tau)$, and $T_{ij}(\tau)^k$

            if(newlag) < len(trajectory):

                # calculate $T_{ij}(k\tau)$ from the data
                
                observed_T = count_transition_matrix(trajectory= trajectory,lag=newlag)[2]
                #observed_T = count_transition_matrix(trajectory= trajectory,lag=TOCOMPLETE)[2]

                # calculate $T_{ij}(\tau)^k$ from the MSM model
                predicted_T = T
                for i in np.arange(k-1):    
                    predicted_T = np.matmul(T,predicted_T)

                # plot the two matrices on the plot.
                for ai in  np.arange(n):
                    for aj in  np.arange(n):
                        axes[ai,aj].scatter(x=newlag,y=predicted_T[ai][aj],marker='.',c='b')
                        axes[ai,aj].scatter(x=newlag,y=observed_T[ai][aj],marker='.',c='r')


        # add some indications and legends
        for ai in  np.arange(n):
            axes[ai,n-1].set_xlabel(f'time lag')
            for aj in  np.arange(n):
                axes[ai,aj].text(kmax*0.8,0.5,f'{ai} -> {aj}')
        axes[-1,-1].legend(['predicted','observed'] )
        plt.title(f'lag = {lag}')
        plt.show()

        # save and close
        fig.savefig(f'{outdir}/{chosen_label}_CKtest_ncluster{n}_lag{lag}.png')
        plt.close()


In [ ]:
trajectory=mytraj3
laglist = np.arange(1,2)
CKtest(trajectory,laglist,outdir)#%%

## Redo the same using another dihedral.
We propose to try it with `chosen_label = 'PSI 0 LEU 2'` using either 2, 3 or 4 clusters.
As previously, you can wrap the data into $[-2,-2+2*\pi]$.

In [ ]:
# after this cell, we focus one dihedral defined by its label, called "chosen_label"

chosen_label = 'PSI 0 LEU 2'
#chosen_label = 'DIH: ALA 3 HA 47 - ALA 3 CA 46 - ALA 3 CB 48 - ALA 3 HB1 49 '
#chosen_label = TOCOMPLETE

chosen_index =  torsions_feat.describe().index(chosen_label)
chosen_label = chosen_label.replace(' ','').replace(':','.')

# selection the torsion data for the chosen label
tor = torsions_data[:,chosen_index]

# plot the chosen data again
fig, axes = plt.subplots(1, 1, figsize=(12, 4), sharex=True)
x = timestep*np.arange(len(tor))
axes.plot(x, tor)
axes.set_ylabel('Wraped dihedral angle / radian')
axes.set_xlabel('time / ns')
axes.set_title(f'{chosen_label} as a function of time')
fig.savefig(f"""{outdir}/{chosen_label}_timeserie.png""")


# apply the wrap function to the torsion data.
wrap_tor = [ wrap_angle(a,-2,2*np.pi) for a in tor] 

# plot the time evolution 
fig, axes = plt.subplots(1, 1, figsize=(12, 4), sharex=True)
x = timestep*np.arange(len(tor))
axes.plot(x, wrap_tor)
axes.set_ylabel('Wraped dihedral angle / radian')
axes.set_xlabel('time / ns')
axes.set_title(f'{chosen_label} as a function of time (wraped)')
fig.savefig(f"""{outdir}/{chosen_label}_timeserie_wraped.png""")
plt.show()
plt.close()

# plot the distribution
nbins = 100
step = (np.max(wrap_tor)-np.min(wrap_tor))/nbins
bins = np.arange(np.min(wrap_tor),np.max(wrap_tor),step)

fig, axes = plt.subplots(1, 1, figsize=(6, 4), sharex=True)
plt.hist(wrap_tor,bins=bins, density=True)
axes.set_xlabel(f'{chosen_label} values (wraped)')
axes.set_title(f'Probability distribution')
fig.savefig(f"""{outdir}/{chosen_label}_histogram_wraped.png""")
plt.show()
plt.close()

Since the peaks are less obvious to separate, we use an automatic clustering algorithm. 

Both the k-means and k-medoids algorithms are partitional. 
They break the dataset up into groups and attempt to minimize the distance between 
points labeled to be in a cluster and a point designated as the center of that cluster.

### The k-medoids algorithm

The k-medoids algorithm works as follows : 
1. Randomly choose k conformations as the ini-
tial cluster centers.

2. Assign each data point to the closest center.

3. For each cluster $C$, propose a random data
point $z \in C$ as the new center  and
evaluate the change using 

$\sum_{x_i \in C} d(x_i , z)^2$

If the newly proposed center reduces the objective function compared to the previous cen-
ter, then replace the current cluster center with $z$.

4. Repeat steps 2 and 3 for a specified number of
iterations or until the algorithm converges to a
stable result.


One advantage of k-medoids is that the resulting centers are actually representative of the data
assigned to them because they lie at the center of
the cluster. A disadvantage is that the number of
clusters must be chosen a priori, compared to k-
centers where it is possible to choose a physically
meaningful criterion for determining the number
of states.

In [ ]:
def clusterize_kmedoids(data, ncluster, maxiter_global=10, maxiter_within=100 ):
    """ 
    inputs :
        data = list of values
        ncluster = integer , number of cluster

    inputs optional :
        maxiter_global=10  ; number of itteration to assign the data to cluster center
        maxiter_within=100  ; number of itteration to optimize a cluster center

    outputs :
        cluster_indices :list of the indices of the cluster centers within data, of length ncluster
        cluster_positions : list of the cluster  centers, of length ncluster
        cluster_sizes : list of the cluster  centers, of length ncluster
        clusternum:  list of the indices for each data , of the same shape as data
    """

    ndata = len(data)

    cluster_indices = [] 
    cluster_positions = []
    cluster_sizes = np.zeros(ncluster,dtype=int)
    cluster_area = np.zeros(ncluster)
    clusternum = [None] * ndata

    if ncluster >= ndata :
        return None

    # randomly choose initial clusters
    for nc in np.arange(ncluster):
        initial = np.random.randint(0,ndata)
        if initial not in cluster_indices:
            cluster_indices.append(initial)
            cluster_positions.append(data[initial])
    #print(f'random initial clusters  indices = {cluster_indices} ')
    #print(f'random initial clusters  positions = {cluster_positions} ')

    # assign each data point to its closest center
    cluster_sizes = np.zeros(ncluster)
    for i in np.arange(ndata):
        distances = np.zeros(ncluster)
        distances =  [ np.abs(data[i]-cp) for cp in cluster_positions ]
        icmin = np.argmin(distances)
        clusternum[i] = icmin
        cluster_sizes[icmin] += 1
    #print(f'clusters  sizes = {cluster_sizes} ')

    # iterate for optimization
    for j in np.arange(maxiter_global):
        #print(f'New assignement of data among cluster {j}/{maxiter_global}')

        # assign each data point to its closest center
        cluster_sizes = np.zeros(ncluster)
        for id in np.arange(ndata):
            distances = np.zeros(ncluster)
            distances =  [ np.abs(data[id]-cp) for cp in  cluster_positions ]
            icmin = np.argmin(distances)
            clusternum[id] = icmin
            cluster_sizes[icmin] += 1
        #print(f'clusters  sizes = {cluster_sizes} ')

        for nc in np.arange(ncluster):
            thisclusterindices = [ii for ii,ic in enumerate(clusternum) if ic == nc]

            # Calculate the optimizing function
            area = 0.0
            for i in thisclusterindices:
                area += (data[i]-cluster_positions[nc])**2
            cluster_area[nc] = area

            # try to find an optimum center for this cluster
            #print(f'this cluster indices = {thisclusterindices}')
            for ii in np.arange(maxiter_within):
                #print(f'optimization within cluster {nc} : {ii}/{maxiter_within}')
                #print(f' area : {cluster_area[nc]}')

                newcenter_index = thisclusterindices[np.random.randint(0,len(thisclusterindices))]
                newcenter_position = data[newcenter_index]

                newarea = 0.0 
                for i in thisclusterindices:
                    newarea += (data[i]-newcenter_position)**2

                if newarea < cluster_area[nc] :
                    cluster_indices[nc] = newcenter_index
                    cluster_positions[nc] = newcenter_position
                    cluster_area[nc] = newarea
                    #print(f'index = {cluster_indices[nc]},pos = {cluster_positions[nc]} is the new center of cluster {nc}, new area = {newarea}')

    return(cluster_indices,cluster_positions,cluster_sizes,clusternum)


In the following cell, the kmedoids algorihm is used to perform the clustering.

**QUESTIONS**  

</font>

1. Complete the following cell on the lines with `TOCOMPLETE`  to do a first clustering with 2 clusters and perform a CK test for a lag time of 1
2. Try with other number of clusters, and with other lag times.
3. Do you find a comnination of {cluster number, lag time} for which the MSM well reproduce the MD data ?

In [ ]:

########### do the clustering

ncluster = 2
#ncluster = TOCOMPLETE

cluster_method = 'k-medoid' 
cluster_indices,cluster_positions,cluster_sizes,clusternum = clusterize_kmedoids(wrap_tor,ncluster)

############ plot the distribution of cluster and their center positions
mytraj =  clusternum


fig, axes = plt.subplots(1, 1, figsize=(6,4), sharex=True)

# define the bons for the distribution
nbins = 100
step = (np.max(wrap_tor)-np.min(wrap_tor))/nbins
bins = np.arange(np.min(wrap_tor),np.max(wrap_tor),step)

# plot the distribution for each cluster
for nc in range(ncluster):
    selected = [t for t,c in zip(wrap_tor,mytraj) if c == nc]
    if len(selected) > 0 :
        plt.hist(selected,bins=bins, density=False,label=f'cluster {nc}',alpha=0.5)
    plt.axvline(x=cluster_positions[nc])
    #plt.axvline(x=TOCOMPLETE)

axes.set_ylabel(torsions_feat.describe()[chosen_index])
axes.set_xlabel(f'{chosen_label} values')
axes.set_title(f'Distribution  in {ncluster} clusters, using {cluster_method} algorithm')
fig.legend()
fig.savefig(f"""{outdir}/{chosen_label}_histogram_attributed_ncluster{ncluster}_method{cluster_method}.png""")
plt.show()
plt.close()


############ look at the transition matrix for lag 1
trajectory = mytraj
lag = 1
#lag = TOCOMPLETE
print(f' for lag = {lag}, transition matrix is ')
print(count_transition_matrix(trajectory,lag)[2])


############ make CK test for various lag times
trajectory=mytraj

laglist = np.arange(1,2)
#laglist = [TOCOMPLETE]
CKtest(trajectory,laglist,outdir)#%%

In [ ]:
# OTIONAL try also with a clustering by hand with 3 clusters. 
# For which time lag do you find a reasonable Markovian process ?

# clusterize 
mylimit1 = 0.5
mylimit2 = 1.2
mytraj=  attribute_3cluster(wrap_tor,mylimit1,mylimit2)


# make CK tests for different lag times
trajectory=mytraj
laglist = np.arange(1,10,3)
CKtest(trajectory,laglist,outdir)#%%